# MoMo Transactions — Practice Task

**Objective:** Clean the transaction data, investigate transaction behaviour, and produce one product-focused insight.

### Tasks
1. Find and remove duplicate rows. How many were there?
2. Count missing `session_seconds` values, decide how to handle them, and defend the decision.
3. Identify the `txn_type` with the biggest mean–median gap and explain the real-world implication in one sentence.
4. Produce one chart that could influence a product manager's decision and state the decision below it.

> **Data note:** The original exercise refers to `momo_transactions.csv`. The working file supplied for this analysis is Excel format, so the loading cell supports both `.csv` and `.xlsx`.


## 1. Import libraries

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt


## 2. Load the data

In [ ]:
# The exercise uses momo_transactions.csv.
# The supplied working file is momo_transactions.xlsx, so this cell supports both formats.

csv_path = "momo_transactions.csv"
xlsx_path = "momo_transactions.xlsx"

if os.path.exists(csv_path):
    txn = pd.read_csv(csv_path)
elif os.path.exists(xlsx_path):
    txn = pd.read_excel(xlsx_path)
else:
    raise FileNotFoundError(
        "Place momo_transactions.csv or momo_transactions.xlsx in the same folder as this notebook."
    )

txn.head()


## 3. Initial data check

In [ ]:
print(f"Rows: {txn.shape[0]:,}")
print(f"Columns: {txn.shape[1]:,}")

txn.info()


## 4. Find and remove duplicate rows

In [ ]:
duplicate_count = txn.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count:,}")

txn_clean = txn.drop_duplicates().copy()

print(f"Rows after removing duplicates: {len(txn_clean):,}")


### Result

There were **60 duplicate rows** in the original dataset. After removing them, the dataset contains **60,000 rows**.


## 5. Missing `session_seconds`

In [ ]:
missing_session = txn_clean["session_seconds"].isna().sum()
print(f"Missing session_seconds after duplicate removal: {missing_session:,}")


### Decision: fill missing `session_seconds` with the median

After duplicate removal, **420 `session_seconds` values are missing**.

I will replace the missing values with the **median session duration**. Session duration can contain unusually long or short sessions, and the median gives a more robust estimate of a typical session while preserving the other transaction records.

The median session duration before imputation is **66.6 seconds**.


In [ ]:
session_median = txn_clean["session_seconds"].median()

txn_clean["session_seconds"] = txn_clean["session_seconds"].fillna(session_median)

print(
    "Missing session_seconds after imputation:",
    txn_clean["session_seconds"].isna().sum()
)


## 6. Mean vs median by transaction type

In [ ]:
txn_type_stats = (
    txn_clean
    .groupby("txn_type")["amount"]
    .agg(["mean", "median"])
)

txn_type_stats["gap"] = txn_type_stats["mean"] - txn_type_stats["median"]

txn_type_stats.sort_values("gap", ascending=False)


In [ ]:
largest_gap_type = txn_type_stats["gap"].idxmax()
largest_gap = txn_type_stats.loc[largest_gap_type, "gap"]

print(f"Transaction type with the biggest gap: {largest_gap_type}")
print(f"Mean–median gap: {largest_gap:,.2f}")


### Explanation

**`savings` has the biggest mean–median gap (8,003.26), suggesting that a smaller number of high-value transactions are pulling the average amount above the typical transaction value.**


## 7. Product decision chart

### Question for the product manager

**Which transaction types receive the most customer activity?**

A transaction-volume chart is useful because product teams often need to prioritise improvements where the largest share of customer activity occurs.


In [ ]:
txn_counts = txn_clean["txn_type"].value_counts()

plt.figure(figsize=(9, 5))
plt.bar(txn_counts.index, txn_counts.values)
plt.title("Transaction Volume by Transaction Type")
plt.xlabel("Transaction Type")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Product decision

**Prioritise product reliability and user-experience improvements around high-volume transaction types, starting with `airtime` because it has the highest transaction volume (18,713 transactions).**

This gives the product team a clear starting point for deciding where improvements can affect the largest number of transactions.


## 8. Final findings

In [ ]:
summary = pd.DataFrame({
    "Finding": [
        "Duplicate rows removed",
        "Missing session_seconds after deduplication",
        "Transaction type with largest mean–median gap",
        "Largest mean–median gap"
    ],
    "Result": [
        duplicate_count,
        missing_session,
        largest_gap_type,
        round(largest_gap, 2)
    ]
})

summary


## Conclusion

The dataset was cleaned by removing duplicate records and imputing missing `session_seconds` values with the median. The transaction-type analysis identified the category with the largest difference between average and typical transaction value. Transaction volume was then used to support a product-prioritisation decision.

**Key takeaway:** transaction volume provides a practical starting point for deciding where product improvements may have the widest customer impact.
